In [1]:
import tkinter as tk
import random
import time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300

POSITION_COST = {
    0: 4, 1: 2, 2: 4,
    3: 2, 4: 1, 5: 2,
    6: 4, 7: 2, 8: 4
}
GOAL_POS = {val: divmod(i, 3) for i, val in enumerate(GOAL)}
def manhattan(board):
    dist = 0
    for i, val in enumerate(board):
        if val == 0:
            continue
        r, c = divmod(i, 3)
        gr, gc = GOAL_POS[val]
        dist += abs(r - gr) + abs(c - gc)
    return dist

def step_cost(board):
    return POSITION_COST[board.index(0)]

def get_neighbors(board):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    result = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            temp = list(board)
            temp[idx], temp[ni] = temp[ni], temp[idx]
            result.append(tuple(temp))
    return result

def hill_climbing(start):
    current = start
    path = [current]
    costs = [0]
    total_cost = 0
    nodes = 1
    while current != GOAL:

        current_h = manhattan(current)
        neighbors = get_neighbors(current)
        nodes += len(neighbors)
        best_neighbor = min(
            neighbors,
            key=lambda state: manhattan(state)
        )

        best_h = manhattan(best_neighbor)
        if best_h >= current_h:
            break
        total_cost += step_cost(best_neighbor)
        current = best_neighbor
        path.append(current)
        costs.append(total_cost)

    return path, costs, nodes

def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(
        1 for i in range(len(arr))
        for j in range(i + 1, len(arr))
        if arr[i] > arr[j]
    )
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t

class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle — Steepest-Ascent Hill Climbing")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board = random_board()
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self.playing = False

        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)

        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(
            left,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white",
            highlightthickness=0
        )
        self.canvas.pack()

        self.step_label = tk.Label(
            left, text="Steps:", font=("Arial", 12), bg="white"
        )
        self.step_label.pack(pady=6)

        self.lb_step_cost = tk.Label(
            left, text="Step cost: —",
            bg="white", fg="#7c5cbf", font=("Arial", 10)
        )
        self.lb_step_cost.pack()

        self.lb_heuristic = tk.Label(
            left, text="h(n) = —",
            bg="white", fg="#c05000", font=("Arial", 10)
        )
        self.lb_heuristic.pack()

        self.lb_total_cost = tk.Label(
            left, text="g(n) = 0  |  f(n) = —", bg="white", fg="blue", font=("Arial", 10)
        )
        self.lb_total_cost.pack()

        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=6)

        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step",    width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)

        tk.Label(right, text="Solution (Steepest-Ascent Hill Climbing)", font=("Arial", 14, "bold"), bg="white").pack()

        self.solution_box = tk.Text(right, width=26, height=25, font=("Consolas", 10))
        self.solution_box.pack()

        self.draw()

    def get_direction(self, old_state, new_state):
        diff = new_state.index(0) - old_state.index(0)
        return {-3: "UP", 3: "DOWN", -1: "LEFT", 1: "RIGHT"}.get(diff, "?")

    def current_board(self):
        return self.solution[self.step] if self.solution else self.board

    def draw(self):
        self.canvas.delete("all")
        board = self.current_board()
        blank_idx = board.index(0)
        cost_val = POSITION_COST[blank_idx]
        h_val = manhattan(board)

        pos_colors = {1: "#e8f5e9", 2: "#fff9c4", 4: "#fce4ec"}

        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE

            if val == 0:
                fill = pos_colors[POSITION_COST[i]]
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="#aaa",
                    width=2, dash=(4, 3)
                )
            else:
                wrong = (val != GOAL[i])
                fill = "#ffe0e0" if wrong else "white"
                self.canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="black", width=2
                )
                self.canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val),
                    font=("Arial", 28),
                    fill="red" if wrong else "black"
                )

        self.lb_step_cost.config(text=f"Step cost: {cost_val}")
        self.lb_heuristic.config(text=f"h(n) = {h_val}")

        if self.costs and self.solution:
            g = self.costs[self.step]
            f = g + h_val
            self.lb_total_cost.config(text=f"g(n) = {g}  |  f(n) = {f}")
        else:
            self.lb_total_cost.config(text=f"g(n) = 0  |  f(n) = {h_val}")

    def shuffle(self):
        self.playing = False
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def reset(self):
        self.playing = False
        self.board = GOAL
        self.solution = []
        self.moves = []
        self.costs = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def _reset_labels(self):
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.lb_step_cost.config(text="Step cost: —")
        self.lb_heuristic.config(text="h(n) = —")
        self.lb_total_cost.config(text="g(n) = 0  |  f(n) = —")
        self.solution_box.delete("1.0", tk.END)

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        path, costs, nodes = hill_climbing(self.board)
        ms = (time.perf_counter() - t0) * 1000

        if path is None:
            return

        self.solution = path
        self.costs = costs
        self.moves = [
            self.get_direction(path[i], path[i + 1])
            for i in range(len(path) - 1)
        ]
        self.step = 0

        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(path) - 1}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")

        self.solution_box.delete("1.0", tk.END)

        self.playing = True
        self.animate()

    def _update_solution_box(self):
        if not self.solution or self.step == 0:
            return
        self.solution_box.config(state=tk.NORMAL)
        self.solution_box.delete("1.0", tk.END)
        for i in range(1, self.step + 1):
            m = self.moves[i - 1]
            board_after = self.solution[i]
            sc = POSITION_COST[board_after.index(0)]
            h = manhattan(board_after)
            g = self.costs[i]
            self.solution_box.insert(
                tk.END,
                f"{i:2}. {m:<5} cost={g}(+{sc}) h={h}\n"
            )
        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        last_line = f"{self.step}.0"
        self.solution_box.tag_add("highlight", last_line, f"{self.step}.end")
        self.solution_box.tag_config("highlight", background="#cce5ff")
        self.solution_box.see(tk.END)

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step == 0:
            self.step_label.config(text="START")
        else:
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")

        self._update_solution_box()

        if self.step < len(self.solution) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False

    def next_step(self):
        self.playing = False
        if not self.solution:
            return
        if self.step < len(self.solution) - 1:
            self.step += 1
            self.draw()
            self.step_label.config(text=f"Step {self.step}: {self.moves[self.step - 1]}")
            self._update_solution_box()

if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()